In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
engine = create_engine('postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/procurement_db')

df = pd.read_sql("SELECT * FROM Supplier_Performance", engine)
print(df.shape)
print(df.head)

(777, 11)
<bound method NDFrame.head of         po_id         supplier  order_date delivery_date    item_category  \
0    PO-00001        Alpha_Inc  17/10/2023    25/10/2023  Office Supplies   
1    PO-00002  Delta_Logistics  25/04/2022    05/05/2022  Office Supplies   
2    PO-00003         Gamma_Co  26/01/2022    15/02/2022              MRO   
3    PO-00004    Beta_Supplies  09/10/2022    28/10/2022        Packaging   
4    PO-00005  Delta_Logistics  08/09/2022    20/09/2022    Raw Materials   
..        ...              ...         ...           ...              ...   
772  PO-00773  Delta_Logistics  11/02/2023    20/02/2023      Electronics   
773  PO-00774        Alpha_Inc  23/07/2022           NaN  Office Supplies   
774  PO-00775         Gamma_Co  19/03/2022    07/04/2022    Raw Materials   
775  PO-00776    Beta_Supplies  30/08/2023    11/09/2023              MRO   
776  PO-00777    Beta_Supplies  09/12/2023    11/12/2023  Office Supplies   

    order_status  quantity  unit_pr

In [3]:
# Convert dates, then calculate delivery time
df["order_date"] = pd.to_datetime(df["order_date"])
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
df["days_to_deliver"] = (df["delivery_date"] - df["order_date"]).dt.days

# Calculate defect rate; avoid division by zero
df["defect_rate"] = (
    df["defective_units"] / df["quantity"].replace(0, np.nan)
) * 100

# Create analysis flags
incomplete_statuses = ["Partially Delivered", "Pending", "Cancelled"]
df["is_incomplete"] = df["order_status"].isin(incomplete_statuses).astype(int)
df["is_compliant"] = (df["compliance"] == "Yes").astype(int)

print(
    df[
        [
            "supplier",
            "days_to_deliver",
            "defect_rate",
            "is_incomplete",
            "is_compliant",
        ]
    ].head()
)

          supplier  days_to_deliver  defect_rate  is_incomplete  is_compliant
0        Alpha_Inc              8.0          NaN              1             1
1  Delta_Logistics             10.0    15.573227              0             1
2         Gamma_Co             20.0     4.505495              0             1
3    Beta_Supplies             19.0     8.333333              0             1
4  Delta_Logistics             12.0    14.491525              0             0


C:\Users\HP\AppData\Local\Temp\ipykernel_16180\4189404322.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["order_date"] = pd.to_datetime(df["order_date"])
C:\Users\HP\AppData\Local\Temp\ipykernel_16180\4189404322.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["delivery_date"] = pd.to_datetime(df["delivery_date"])


In [10]:
scorecard = df.groupby('supplier').agg({
    'po_id': 'count',
    'quantity': 'sum',
    'defective_units': 'sum',
    'days_to_deliver': 'mean',
    'is_compliant': 'mean',
    'is_incomplete': 'mean'
}).round(2)

scorecard.columns = [
    'Total_Orders', 'Total_Quantity', 'Total_Defects',
    'Avg_Delivery_Days', 'Compliance_Rate', 'Incomplete_Rate'
]

scorecard['Defect_Rate'] = (
    scorecard['Total_Defects'] / scorecard['Total_Quantity'] * 100
).round(2)
scorecard['Compliance_Rate'] = (scorecard['Compliance_Rate'] * 100).round(2)
scorecard['Incomplete_Rate'] = (scorecard['Incomplete_Rate'] * 100).round(2)

scorecard = scorecard.sort_values('Defect_Rate')
print(scorecard)

                 Total_Orders  Total_Quantity  Total_Defects  \
supplier                                                       
Alpha_Inc                 141          150414         2717.0   
Epsilon_Group             166          180053         4682.0   
Gamma_Co                  143          157916         7034.0   
Beta_Supplies             156          180516        13838.0   
Delta_Logistics           171          181652        19678.0   

                 Avg_Delivery_Days  Compliance_Rate  Incomplete_Rate  \
supplier                                                               
Alpha_Inc                    10.61             94.0             24.0   
Epsilon_Group                10.87             98.0             28.0   
Gamma_Co                     10.19             86.0             28.0   
Beta_Supplies                11.27             76.0             29.0   
Delta_Logistics              10.85             61.0             30.0   

                 Defect_Rate  
supplier       

In [12]:
corr = df[['days_to_deliver', 'defect_rate', 'is_incomplete', 'is_compliant', 'quantity']].corr().round(3)
print(corr)

                 days_to_deliver  defect_rate  is_incomplete  is_compliant  \
days_to_deliver            1.000        0.031          0.078         0.034   
defect_rate                0.031        1.000          0.049        -0.309   
is_incomplete              0.078        0.049          1.000        -0.021   
is_compliant               0.034       -0.309         -0.021         1.000   
quantity                   0.024       -0.071         -0.005         0.067   

                 quantity  
days_to_deliver     0.024  
defect_rate        -0.071  
is_incomplete      -0.005  
is_compliant        0.067  
quantity            1.000  


## Insights

1. Alpha_Inc is the best supplier overall. It has the lowest defect rate (1.81%), compliance of 94% and relatively few incomplete orders.
2. Epsilon_Group is very reliable. It has the highest compliance rate (98%) and a low defect rate (2.60%), which makes it a strong second choice.
3. Delta_Logistics carries the most risk. It has the highest defect rate (10.83%), the lowest compliance (61%) and the most incomplete orders.
4. Beta_Supplies also falls short. Its defect rate is high (7.67%) and compliance is low (75.64%), so it needs attention.
5. Delivery speed is about the same across suppliers. Average delivery time runs from 10.2 to 11.3 days, so quality and compliance are what set them apart.
6. The overall defect rate is 5.64% and compliance is 82.4%, so supplier quality management has plenty of room to improve.

## Recommendations

1. Give Alpha_Inc and Epsilon_Group the critical and high-volume orders.
2. Put Delta_Logistics under performance review. Cut its order volume or look for alternative suppliers.
3. Work with Beta_Supplies on quality control and compliance.
4. Set clear supplier SLAs:
   - Defect rate below 3%
   - Compliance above 90%
5. Start regular supplier scorecards covering quality, compliance and fulfilment.
6. Look into the root causes of high defects and incomplete orders, especially at Delta and Beta.

In [15]:
# 1. Supplier Scorecard
scorecard.to_csv("supplier_scorecard.csv")

# 2. Order Status
order_status = df['order_status'].value_counts(normalize=True).mul(100).round(2).reset_index()
order_status.columns = ['order_status', 'Percentage']
order_status.to_csv("order_status.csv", index=False)

# 3. Overall KPIs
kpis = pd.DataFrame({
    'Metric': ['Total Orders', 'Overall Defect Rate (%)', 'Overall Compliance Rate (%)', 'Avg Delivery Days'],
    'Value': [777, 5.64, 82.37, 10.78]
})
kpis.to_csv("kpis.csv", index=False)

print("Files exported successfully!")

Files exported successfully!
